In [0]:
# Reads from staging Delta tables
# Writes Parquet snapshots to chinook volume
# Logs results to execution_log

dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("staging_schema", "")
dbutils.widgets.text("raw_schema", "")
dbutils.widgets.text("volume_name", "")
dbutils.widgets.text("metadata_schema", "")
dbutils.widgets.text("table_name", "")

catalog_name     = dbutils.widgets.get("catalog_name")
staging_schema   = dbutils.widgets.get("staging_schema")
raw_schema       = dbutils.widgets.get("raw_schema")
volume_name      = dbutils.widgets.get("volume_name")
metadata_schema  = dbutils.widgets.get("metadata_schema")
table_name       = dbutils.widgets.get("table_name")

print(f"Processing: {table_name}")

In [0]:
# Step 1: Read from staging

from datetime import datetime

staging_path = f"{catalog_name}.{staging_schema}.{table_name.lower()}"
df = spark.read.table(staging_path)
source_count = df.count()

print(f"Read {source_count} rows from {staging_path}")

In [0]:
# Step 2: Build dynamic file path
# Structure: /Volumes/<catalog>/raw_zone/chinook/<table>/<yyyy>/<mm>/<dd>/<table>.parquet
# New snapshot every run, never overwrite

from datetime import datetime

now = datetime.now()
EXTRACT_DATE = datetime.now().strftime("%Y-%m-%d")

file_path = f"/Volumes/{catalog_name}/{raw_schema}/{volume_name}/{table_name.lower()}/{EXTRACT_DATE}/{table_name.lower()}.parquet"

print(f"Writing to: {file_path}")

In [0]:
# Step 3: Write to volume as Parquet

df.write.mode("overwrite").parquet(file_path)

print(f"Parquet written to {file_path}")

In [0]:
# Step 4: Validate row counts
# Pipeline only valid when source == target

target_df    = spark.read.parquet(file_path)
target_count = target_df.count()

if source_count == target_count:
    status = "SUCCESS"
    print(f"Row count validated: {source_count} rows")
else:
    status = "FAILED"
    print(f"Mismatch! Source: {source_count} | Target: {target_count}")

In [0]:
# Step 5: Log to execution_log
# One row per table per run

from pyspark.sql import Row

log_row = spark.createDataFrame([Row(
    table_name       = table_name,
    execution_time   = datetime.now(),
    status           = status,
    source_row_count = source_count,
    target_row_count = target_count,
    file_location    = file_path,
    created_date     = datetime.now().date()
)])

log_row.write.format("delta") \
    .mode("append") \
    .saveAsTable(f"{catalog_name}.{metadata_schema}.execution_log")

print(f"Logged to execution_log — Status: {status}")

In [0]:
# Step 6: Fail the task if validation failed
# Stops the Job from proceeding to Bronze

if status == "FAILED":
    raise Exception(f"Row count mismatch for {table_name}. Check execution_log.")

In [0]:
#verification of execution log and raw parquet files in chinook volume

print("Execution log:")
display(spark.sql(f"SELECT * FROM {catalog_name}.{metadata_schema}.execution_log ORDER BY execution_time DESC"))

print(f"\nRaw volume files for {table_name}:")
display(dbutils.fs.ls(f"/Volumes/{catalog_name}/{raw_schema}/{volume_name}/{table_name.lower()}/"))